# Relative Frequency by CPT Code (2024)

This notebook computes the relative frequency of services by CPT/HCPCS (CPT) code using the `Physician_Supplier_Procedure_Summary_2024.csv` file.

We will:
- Load the large CSV in chunks with pandas
- Aggregate total submitted service counts by `HCPCS_CD`
- Compute each code's share of all services (relative frequency)
- Save the result to a smaller CSV for further analysis.

In [1]:
import pandas as pd

# Path to the big CSV file with physician/supplier data
csv_path = r"c:\Users\alece\OneDrive\MIDS Program\Supplier Procedure Summary\Physician\Supplier Procedure Summary\2024\Physician_Supplier_Procedure_Summary_2024.csv"

# We only need these columns for relative frequency by CPT/HCPCS code
usecols = ["HCPCS_CD", "PSPS_SUBMITTED_SERVICE_CNT"]

csv_path

'c:\\Users\\alece\\OneDrive\\MIDS Program\\Supplier Procedure Summary\\Physician\\Supplier Procedure Summary\\2024\\Physician_Supplier_Procedure_Summary_2024.csv'

In [4]:
# Compute total submitted services and relative frequency per CPT/HCPCS code

totals = {}

# Process the CSV in chunks so it can fit in memory
for chunk in pd.read_csv(csv_path, usecols=usecols, chunksize=500_000):
    # Drop rows without a code
    chunk = chunk.dropna(subset=["HCPCS_CD"])

    # Sum services per code within this chunk
    chunk["PSPS_SUBMITTED_SERVICE_CNT"] = pd.to_numeric(
    chunk["PSPS_SUBMITTED_SERVICE_CNT"], errors="coerce"
)

    grp = chunk.groupby("HCPCS_CD")["PSPS_SUBMITTED_SERVICE_CNT"].sum()

    # Accumulate into a single totals dict
    for code, cnt in grp.items():
        totals[code] = totals.get(code, 0) + cnt

# Convert to a Series/DataFrame
service_counts = pd.Series(totals, name="service_count").sort_index()

# Total number of services across all codes
total_services = service_counts.sum()

# Relative frequency for each CPT/HCPCS code
relative_frequency = (service_counts / total_services).rename("relative_frequency")

# Combine into one result table and sort by relative frequency (descending)
result = pd.concat([service_counts, relative_frequency], axis=1).sort_values(
    "relative_frequency", ascending=False
)

# Save to a smaller CSV for further analysis
output_path = "hcpcs_relative_frequency_2024.csv"
result.to_csv(output_path)

# Show the top 10 most frequent codes
output_path, result.head(10)

('hcpcs_relative_frequency_2024.csv',
        service_count  relative_frequency
 J7677   1.012679e+09            0.151725
 A4353   1.974804e+08            0.029588
 B4152   1.290424e+08            0.019334
 J7613   1.195071e+08            0.017905
 A0425   1.115785e+08            0.016717
 99214   1.077434e+08            0.016143
 Q9967   1.042806e+08            0.015624
 J1010   9.005339e+07            0.013492
 J0717   7.837800e+07            0.011743
 J7507   7.810417e+07            0.011702)

In [5]:
# Load the precomputed relative frequency table from CSV

rel_path = "hcpcs_relative_frequency_2024.csv"

rel_df = pd.read_csv(rel_path, index_col=0)
rel_df.index.name = "HCPCS_CD"

# Preview the first few rows to confirm
rel_df.head(10)

,service_count,relative_frequency
HCPCS_CD,,
J7677,1.012679e+09,0.151725
A4353,1.974804e+08,0.029588
B4152,1.290424e+08,0.019334
J7613,1.195071e+08,0.017905
A0425,1.115785e+08,0.016717
99214,1.077434e+08,0.016143
Q9967,1.042806e+08,0.015624
J1010,9.005339e+07,0.013492
J0717,7.837800e+07,0.011743


In [6]:
codes_of_interest = [
    "77067",
    "77065",
    "77066",
    "77063",
    "77061",
    "77062",
    "77046",
    "77047",
    "77048",
    "77049"
]

In [7]:
subset = rel_df.reindex(codes_of_interest)
subset

,service_count,relative_frequency
HCPCS_CD,,
77067,6274834.5,9.401276e-04
77065,737740.1,1.105320e-04
77066,602064.0,9.020429e-05
77063,6007812.0,9.001209e-04
77061,1327.0,1.988179e-07
77062,1579.0,2.365738e-07
77046,32.0,4.794402e-09
77047,2253.0,3.375559e-07
77048,168.0,2.517061e-08
